# Q1 and Q2, step by step

The two questions:

- Q1: do authors return to journals they already published in, more often than chance?
- Q2: do authors stay with the same publisher, more often than chance?

Both answers have the same shape: one number = observed rate divided by expected rate under a chance model. 1.0 means chance explains everything, above 1 means real attachment.

The steps in this notebook:

1. load the data
2. build each author's publication history
3. cross-check our counting against Lennart's Prolog output
4. compute the observed rates
5. build the chance model (two versions)
6. results
7. why Q3 is not in this notebook

Everything is standard library, seed 42, full run takes about half a minute.

## Step 1: load the data

Two inputs:

- the semiclean CSV (local in `../data/`, not in git): one row per paper with date, journal, year, topic, and the author list
- Lennart's journal-to-publisher mapping from `../results/` in the repo, so we can roll journals up to their parent publisher (BMC counts as Springer Nature)

One cleanup: a few papers list the same author twice, we keep each author once per paper.

In [1]:
import csv, json, random, sys
from bisect import bisect
from collections import defaultdict

csv.field_size_limit(sys.maxsize)   # the authorships json column is longer than the csv default allows

works = {}          # work_id -> (date, journal, year, topic)
work_authors = {}   # work_id -> [author_id, ...] without duplicates
with open("../data/openalex_ai_semiclean_v1_0.csv", encoding="utf-8-sig", newline="") as f:
    for r in csv.DictReader(f):
        w = r["work_id"]
        works[w] = (r["publication_date"], r["journal_id"], int(r["publication_year"]), r["primary_topic_id"])
        seen = set(); auths = []
        for a in json.loads(r["authorships_json"]):
            if a.get("author_id") and a["author_id"] not in seen:   # some papers list an author twice, keep one
                seen.add(a["author_id"]); auths.append(a["author_id"])
        work_authors[w] = auths

parent = {}   # journal -> parent publisher, None when unresolved (1 journal)
with open("../results/openalex_three_path_v1_0/journal_parent_publishers.csv", newline="", encoding="utf-8-sig") as f:
    for r in csv.DictReader(f):
        parent[r["journal_id"]] = None if r["is_unresolved"] == "True" else r["parent_publisher_id"]

print(f"{len(works):,} papers, {len(parent)} journals mapped to publishers")

27,400 papers, 64 journals mapped to publishers


## Step 2: build each author's publication history

Everything below asks some version of "had this author already done X before this paper?", so we need each author's papers in time order. A paper with 3 authors lands in 3 histories, which is where the author-paper pairs come from (same unit as Lennart's run). One rule to keep in mind: only strictly earlier dates count as before. Two papers on the same date do not see each other, because OpenAlex fills missing day and month with January 1 and the order within such a date is unknowable.

In [2]:
papers_of = defaultdict(list)   # author -> [(date, work, journal)]
for w, (d, j, y, t) in works.items():
    for a in work_authors[w]:   # every author of a paper gets their own row
        papers_of[a].append((d, w, j))
for a in papers_of:
    papers_of[a].sort()   # dates are ISO strings, sorting puts earlier papers first

n = sum(len(v) for v in papers_of.values())
print(f"{len(papers_of):,} authors, {n:,} author-paper pairs")

82,539 authors, 110,654 author-paper pairs


## Step 3: cross-check against the Prolog output

Before computing anything new, we make sure we count the same way Lennart does. We rebuild his three flags in our own code and compare every row:

- journal_path: the author already published in this journal
- publisher_path: the author already published with this parent publisher through at least one other journal
- coauthor_path: someone else on this paper already published in this journal

If all 110,654 rows agree, two independent implementations confirm each other, and we can trust the flags. This is the same idea as the Prolog-vs-Python parity check in the project plan (S2.4), done from our side.

In [3]:
# walk each author's papers in date order and remember what was seen before
my = {}  # (work, author) -> [journal_path, publisher_path, coauthor_path]
for a, lst in papers_of.items():
    seen_j = set()               # journals seen at earlier dates
    seen_pj = defaultdict(set)   # parent publisher -> journals it was seen through
    i = 0
    while i < len(lst):
        d0 = lst[i][0]; grp = []
        while i < len(lst) and lst[i][0] == d0:   # papers sharing a date form one group
            grp.append(lst[i]); i += 1
        for d, w, j in grp:                        # flags first, against strictly earlier papers only
            p = parent.get(j)
            my[(w, a)] = [j in seen_j,
                          p is not None and p in seen_pj and any(x != j for x in seen_pj[p]),   # parent seen through another journal
                          False]
        for d, w, j in grp:                        # then the group becomes history
            seen_j.add(j)
            p = parent.get(j)
            if p is not None:
                seen_pj[p].add(j)

# coauthor flag in two passes
first_in_journal = {}   # (author, journal) -> earliest date they appeared there
for a, lst in papers_of.items():
    for d, w, j in lst:
        k = (a, j)
        if k not in first_in_journal or d < first_in_journal[k]:
            first_in_journal[k] = d
for w, (d, j, y, t) in works.items():
    # "9999" sorts after every real date, so missing means never
    early = [b for b in work_authors[w] if first_in_journal.get((b, j), "9999") < d]   # authors of this paper who were in the journal before
    for a in work_authors[w]:
        my[(w, a)][2] = any(b != a for b in early)   # at least one of them must be someone else

In [4]:
lf = {}   # the same flags as Prolog computed them
with open("../results/openalex_three_path_v1_0/pathway_flags.csv", newline="", encoding="utf-8-sig") as f:
    for r in csv.DictReader(f):
        lf[(r["focal_work_id"], r["focal_author_id"])] = (
            r["journal_path"] == "True", r["publisher_path"] == "True", r["coauthor_path"] == "True")

assert set(my) == set(lf)   # both sides must cover exactly the same pairs
mismatches = [k for k in lf if tuple(my[k]) != lf[k]]
cnt = lambda d, i: sum(1 for v in d.values() if v[i])   # how many pairs have flag number i
print(f"journal {cnt(my,0):,} vs {cnt(lf,0):,} | publisher {cnt(my,1):,} vs {cnt(lf,1):,} | coauthor {cnt(my,2):,} vs {cnt(lf,2):,}")
print(f"mismatching rows: {len(mismatches)}")
assert not mismatches and cnt(my,0) == 12325 and cnt(my,1) == 4919 and cnt(my,2) == 26136   # pinned to his run, drift fails loudly
print("every row matches, the flags are confirmed")

journal 12,325 vs 12,325 | publisher 4,919 vs 4,919 | coauthor 26,136 vs 26,136
mismatching rows: 0
every row matches, the flags are confirmed


## Step 4: the observed rates

Three statistics, each as a share of all author-paper pairs:

- Q1 observed: the author was already in this journal. Example: Anna publishes in Sensors, and she had a Sensors paper two years ago, that pair counts
- Q2 wide: the author was already with this parent publisher, no matter through which journal. Note this includes essentially every Q1 case, same journal means same publisher (essentially because 6 Q1 pairs sit on the one journal without a resolvable publisher and drop out here)
- Q2 narrow: the author was already with this parent publisher through at least one other journal. This is the version that asks about the publisher and not the journal. It does not require this journal to be new to the author though, 1,468 of the 4,919 narrow pairs are journal returns as well

We need all three because Q2 wide alone cannot tell publisher loyalty apart from journal loyalty.

In [5]:
# same walk as step 3, now counting the three statistics instead of storing flags
q1_pos = q2w = q2n = 0
eligible = 0        # pairs that had any earlier paper at all, only those can repeat
q1_eligible = 0     # of those, how many are returns
overlap = 0         # narrow pairs that are also Q1 returns
for a, lst in papers_of.items():
    seen_j = set(); seen_p = set(); seen_pj = defaultdict(set); i = 0
    history = 0     # how many papers of this author lie strictly earlier
    while i < len(lst):
        d0 = lst[i][0]; grp = []
        while i < len(lst) and lst[i][0] == d0:
            grp.append(lst[i]); i += 1
        for d, w, j in grp:
            if history: eligible += 1
            f1 = j in seen_j
            p = parent.get(j)
            f2 = p is not None and p in seen_pj and any(x != j for x in seen_pj[p])
            if f1: q1_pos += 1
            if f1 and history: q1_eligible += 1
            if p is not None and p in seen_p: q2w += 1   # parent seen before, same journal allowed
            if f2: q2n += 1   # parent seen through at least one other journal, this one may be known too
            if f1 and f2: overlap += 1
        for d, w, j in grp:
            history += 1
            seen_j.add(j); p = parent.get(j)
            if p is not None: seen_p.add(p); seen_pj[p].add(j)

obs = (q1_pos / n, q2w / n, q2n / n)
print(f"Q1 observed:        {obs[0]:.4f}  ({q1_pos:,} of {n:,} pairs)")
print(f"Q2 wide observed:   {obs[1]:.4f}  ({q2w:,} pairs)")
print(f"Q2 narrow observed: {obs[2]:.4f}  ({q2n:,} pairs)")
print(f"\nof the {q2n:,} narrow pairs, {overlap:,} are journal returns as well, {q2n - overlap:,} are not")
print(f"decomposition: {q1_pos:,} Q1 + {q2n - overlap:,} narrow-not-Q1 - 6 on the unresolved journal = {q2w:,} wide")
print(f"\n{eligible:,} of {n:,} pairs ({eligible/n:.0%}) had an earlier paper at all, the rest cannot repeat by construction")
print(f"among those eligible pairs Q1 is {q1_eligible/eligible:.0%}, not {obs[0]:.0%}")

Q1 observed:        0.1114  (12,325 of 110,654 pairs)
Q2 wide observed:   0.1425  (15,770 pairs)
Q2 narrow observed: 0.0445  (4,919 pairs)

of the 4,919 narrow pairs, 1,468 are journal returns as well, 3,451 are not
decomposition: 12,325 Q1 + 3,451 narrow-not-Q1 - 6 on the unresolved journal = 15,770 wide

26,790 of 110,654 pairs (24%) had an earlier paper at all, the rest cannot repeat by construction
among those eligible pairs Q1 is 46%, not 11%


Two things worth keeping in mind from those prints. First, 11% is a share of all pairs, but 81% of authors have exactly one paper in the corpus and can never repeat. Among the pairs that had an earlier paper at all, Q1 is 46%. The chance model contains the same structural zeros, so the ratio is unaffected, but quote the right number for the right question.

Second, is 11% a lot? No idea yet. If one journal published half the field, plenty of returns would happen by pure volume. That is what the chance model is for.

## Step 5: the chance model

The idea: keep everything about each author fixed (how many papers, in which years, on which dates) and randomize only where each paper landed. Then count the same three statistics on the randomized world, 100 times, and average.

- each paper gets a random journal, drawn with probability proportional to how many corpus papers that journal published in that year. Big journals get drawn often, small ones rarely
- if the observed rate is much higher than the redrawn rate, size alone does not explain the returning

We run the redraw twice, with one difference:

- version A: a paper can land in any journal of that year
- version B: a paper can only land in journals that published its topic in that year. An NLP paper realistically only chooses among NLP journals, and that concentration alone produces repeats. Version B removes exactly that effect
- one caveat for B: OpenAlex assigns topics partly based on the venue, so stratifying by topic can also absorb a bit of genuine journal attachment. Realistically the truth lies between A and B

In [6]:
def build_dist(keyfun):
    """journal size table per cell (cell = year, or year+topic)"""
    cells = defaultdict(lambda: defaultdict(int))
    for w, tup in works.items():
        cells[keyfun(tup)][tup[1]] += 1
    out = {}
    for k, cts in cells.items():
        js, wts = zip(*sorted(cts.items()))
        cum = []; s = 0
        for x in wts:
            s += x; cum.append(s)   # cumulative sums so bisect can draw a weighted random journal
        out[k] = (js, cum, s)
    return out

def null_rates(dist, keyfun, M=100):
    """redraw all journals M times, return average rates for Q1, Q2 wide, Q2 narrow"""
    rng = random.Random(42)   # fresh seed per version so each reproduces on its own
    acc = [0.0, 0.0, 0.0]
    for _ in range(M):
        rep = pw = pn = 0
        for a, lst in papers_of.items():
            seen_j = set(); seen_p = set(); seen_pj = defaultdict(set); i = 0
            while i < len(lst):
                d0 = lst[i][0]; grp = []
                while i < len(lst) and lst[i][0] == d0:
                    grp.append(lst[i]); i += 1
                drawn = []
                for d, w, j in grp:
                    js, cum, s = dist[keyfun(works[w])]
                    dj = js[bisect(cum, rng.random() * s)]   # size-weighted random journal
                    drawn.append(dj)
                    if dj in seen_j: rep += 1
                    p = parent.get(dj)
                    if p is not None:
                        if p in seen_p: pw += 1
                        if p in seen_pj and any(x != dj for x in seen_pj[p]): pn += 1
                for dj in drawn:                     # the date group becomes history afterwards
                    seen_j.add(dj); p = parent.get(dj)
                    if p is not None: seen_p.add(p); seen_pj[p].add(dj)
        acc[0] += rep / n; acc[1] += pw / n; acc[2] += pn / n   # running sums, divided by M below
    return [x / M for x in acc]

dist_year = build_dist(lambda t: t[2])                  # version A, cell = year
dist_topic = build_dist(lambda t: (t[2], t[3]))         # version B, cell = year and topic

# how restrictive is version B? cells with a single journal leave the paper no choice at all
degenerate = [k for k, v in dist_topic.items() if len(v[0]) == 1]
print(f"version B has {len(dist_topic)} year+topic cells, {len(degenerate)} of them contain a single journal "
      f"({sum(dist_topic[k][2] for k in degenerate)} papers, {sum(dist_topic[k][2] for k in degenerate)/len(works):.1%} of the corpus)")

e_year  = null_rates(dist_year, lambda t: t[2])
e_topic = null_rates(dist_topic, lambda t: (t[2], t[3]))
print(f"expected under A (year):        Q1 {e_year[0]:.4f}, Q2 wide {e_year[1]:.4f}, Q2 narrow {e_year[2]:.4f}")
print(f"expected under B (year+topic):  Q1 {e_topic[0]:.4f}, Q2 wide {e_topic[1]:.4f}, Q2 narrow {e_topic[2]:.4f}")

version B has 470 year+topic cells, 22 of them contain a single journal (25 papers, 0.1% of the corpus)


expected under A (year):        Q1 0.0361, Q2 wide 0.0706, Q2 narrow 0.0383
expected under B (year+topic):  Q1 0.0501, Q2 wide 0.0843, Q2 narrow 0.0400


## Step 6: results, observed divided by expected

In [7]:
print("                                   A: year null   B: year+topic null")
for i, nm in enumerate(["Q1 journal repeat               ",
                        "Q2 wide (same parent, any)      ",
                        "Q2 narrow (same parent, other)  "]):
    print(f"{nm}   {obs[i]/e_year[i]:.2f}           {obs[i]/e_topic[i]:.2f}")

# pinned so a broken rerun fails instead of printing wrong numbers
assert abs(obs[0]/e_year[0] - 3.09) < 0.05 and abs(obs[0]/e_topic[0] - 2.22) < 0.05   # Q1
assert abs(obs[1]/e_year[1] - 2.02) < 0.05 and abs(obs[1]/e_topic[1] - 1.69) < 0.05   # Q2 wide
assert abs(obs[2]/e_year[2] - 1.16) < 0.05 and abs(obs[2]/e_topic[2] - 1.11) < 0.05   # Q2 narrow
assert (q1_pos, q2w, q2n, overlap, eligible) == (12325, 15770, 4919, 1468, 26790)     # the counts behind the story

                                   A: year null   B: year+topic null
Q1 journal repeat                  3.09           2.22
Q2 wide (same parent, any)         2.02           1.69
Q2 narrow (same parent, other)     1.16           1.11


What the table says:

- Q1 lands between 2.2 and 3.1: authors return to the same journal two to three times more often than chance predicts. Real attachment, robust under both chance models
- Q2 wide (2.0 / 1.7) looks like publisher loyalty but is not: it inherits most of its size from Q1, because returning to the same journal always means the same publisher
- Q2 narrow is the honest publisher question, and it lands at 1.1 to 1.2: staying with a publisher beyond the journal you already know barely happens more than chance. That is a finding, just not the one we expected

Still open, tracked in the issues: the null keeps author productivity fixed but not the stricter degree-preserving way (#42), the publisher map is today's ownership, Hindawi counts as Wiley for all years (#28), redraw spread is not a confidence interval, real uncertainty needs the author-level bootstrap (#50), and history is only what happened inside the 64 journals from 2015 on.

## Step 7: why Q3 is not in this notebook

Q3 asks whether a co-author connection makes an author enter a journal that is new to them. Answering that needs opportunities, including the journals an author could have entered but did not, and this table only contains published papers. The denominator is missing (#52).

To show what goes wrong if you ignore that, the cell below computes the ratio anyway. It comes out at 0.68, which would mean co-authors prevent first entries. Nonsense, but instructive: papers with a connected co-author sit in journals the author already knows, so selecting on publication flips the sign. And the current coauthor flag means "co-author on this very paper", the riding-along case, while Q3 needs the earlier-collaborator version (PR #49).

In [8]:
f1c1 = sum(1 for v in my.values() if not v[0] and v[2])   # first entries among pairs with the coauthor flag
f1c0 = sum(1 for v in my.values() if not v[0] and not v[2])   # first entries among pairs without it
c1 = cnt(my, 2); c0 = n - c1
print(f"P(first entry | coauthor flag) = {f1c1/c1:.3f}")
print(f"P(first entry | no flag)       = {f1c0/c0:.3f}")
print(f"ratio = {(f1c1/c1)/(f1c0/c0):.3f}  (not Q3, see text above)")

P(first entry | coauthor flag) = 0.656
P(first entry | no flag)       = 0.961
ratio = 0.683  (not Q3, see text above)


## Next

- opportunity set decision (#52), then the event table gives Q3 its denominator
- seed definition and outcome split in the rules (PR #49)
- degree-preserving null once #42 is decided, publication-time publisher map for Q2 (#28)
- report both chance models in the report and explain why they differ
- TODO try a pair-weighted null. 421 papers have no usable author id, they add journal volume but contribute no pairs. Probably moves the expected rates only a little, but untested
- TODO the clean "publisher loyalty into a new journal" statistic (narrow and not Q1, 3,451 pairs) with a matching null. Not computed yet